In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import sys

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
from tqdm.auto import tqdm
import h5py 
import pandas as pd 

## Check format of training H5s

In [2]:
########
# Get paths to uncombined h5
source_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/jsinV3BalancedProcessed/sr_20000/splits/train_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'
noise_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/cullLabels_silence/sr20000_unbalanced_train_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdh5' 
dataframe_metadata = pd.read_hdf(source_h5)
source_files = h5py.File(source_h5, 'r')

noise_metadata = pd.read_hdf(noise_h5)
noise_files = h5py.File(noise_h5, 'r')

In [3]:
train_word_dict = {class_int:word for class_int,word in dataframe_metadata[['word_int', 'word']].values}

train_word_dict = dict(sorted(train_word_dict.items()))
train_word_dict

{0: '__nullSignal__',
 1: 'ability',
 2: 'about',
 3: 'above',
 4: 'accepted',
 5: 'according',
 6: 'account',
 7: 'across',
 8: 'action',
 9: 'active',
 10: 'activities',
 11: 'activity',
 12: 'actually',
 13: 'added',
 14: 'addition',
 15: 'additional',
 16: 'advanced',
 17: 'africa',
 18: 'after',
 19: 'again',
 20: 'against',
 21: 'agency',
 22: 'agreed',
 23: 'agreement',
 24: 'allow',
 25: 'allowed',
 26: 'almost',
 27: 'alone',
 28: 'along',
 29: 'already',
 30: 'alternative',
 31: 'although',
 32: 'always',
 33: 'america',
 34: 'american',
 35: 'americans',
 36: 'among',
 37: 'amount',
 38: 'analysts',
 39: 'ancient',
 40: 'animals',
 41: 'announced',
 42: 'annual',
 43: 'another',
 44: 'appear',
 45: 'appearance',
 46: 'appeared',
 47: 'appears',
 48: 'appointed',
 49: 'approach',
 50: 'approximately',
 51: 'april',
 52: 'areas',
 53: 'around',
 54: 'article',
 55: 'asked',
 56: 'assets',
 57: 'associated',
 58: 'association',
 59: 'attack',
 60: 'attacks',
 61: 'attempt',
 62

In [10]:
source_files['ndarray_data']['signal']

<HDF5 dataset "signal": shape (230357,), type "|O">

### Get val h5s 


In [11]:
########
# Get paths to uncombined h5
val_source_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/jsinV3BalancedProcessed/sr_20000/splits/valid_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'
val_noise_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/cullLabels_silence/sr20000_balanced_train_segments_raw_exclude_speech_and_only_music_maxZerosPercent10.pdh5' 
val_dataframe_metadata = pd.read_hdf(val_source_h5)
val_source_files = h5py.File(val_source_h5, 'r')

val_noise_metadata = pd.read_hdf(val_noise_h5)
val_noise_files = h5py.File(val_noise_h5, 'r')

In [12]:
val_word_dict = {class_int:word for class_int,word in val_dataframe_metadata[['word_int', 'word']].values}

val_word_dict = dict(sorted(val_word_dict.items()))
val_word_dict

{0: '__nullSignal__',
 1: 'ability',
 2: 'about',
 3: 'above',
 4: 'accepted',
 5: 'according',
 6: 'account',
 7: 'across',
 8: 'action',
 9: 'active',
 10: 'activities',
 11: 'activity',
 12: 'actually',
 13: 'added',
 14: 'addition',
 15: 'additional',
 16: 'advanced',
 17: 'africa',
 18: 'after',
 19: 'again',
 20: 'against',
 21: 'agency',
 22: 'agreed',
 23: 'agreement',
 24: 'allow',
 25: 'allowed',
 26: 'almost',
 27: 'alone',
 28: 'along',
 29: 'already',
 30: 'alternative',
 31: 'although',
 32: 'always',
 33: 'america',
 34: 'american',
 35: 'americans',
 36: 'among',
 37: 'amount',
 38: 'analysts',
 39: 'ancient',
 40: 'animals',
 41: 'announced',
 42: 'annual',
 43: 'another',
 44: 'appear',
 45: 'appearance',
 46: 'appeared',
 47: 'appears',
 48: 'appointed',
 49: 'approach',
 50: 'approximately',
 51: 'april',
 52: 'areas',
 53: 'around',
 54: 'article',
 55: 'asked',
 56: 'assets',
 57: 'associated',
 58: 'association',
 59: 'attack',
 60: 'attacks',
 61: 'attempt',
 62

In [13]:
train_word_dict == val_word_dict

True

In [3]:
all_labels = [l for label in noise_metadata.labels_decoded_slugged.values for l in label.split(',')]

In [4]:
np.unique(all_labels)[0]

np.str_('__nullNoise__')

In [5]:
len(noise_metadata)

718625

In [6]:
noise_files['ndarray_data']['labels_binary_via_int'][0].shape

(517,)

In [7]:
noise_metadata

,YTID,_index_before_label_removal,_pre_map_index,_pre_subset_index,_signal_length_samples,_signal_num_zeros,_signal_zeros_ratio,channel_num,corpus,data_split,...,path,removed_labels,signal_length,source,sr,start_YT_clip_secs,start_secs,youtube_source,labels_int,corpus_int
0,dEpyjSmHd58,0.0,0.0,0.0,200157.0,2811.0,0.014044,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,/m/04rlf,441344.0,YT_dEpyjSmHd58.wav,20000,0.0,0.0,http://youtu.be/dEpyjSmHd58?start=0&end=10,[113],0
1,op0cgaKq6Ck,1.0,1.0,1.0,200107.0,0.0,0.000000,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,/m/04rlf,480256.0,YT_op0cgaKq6Ck.wav,20000,140.0,0.0,http://youtu.be/op0cgaKq6Ck?start=140&end=150,"[16, 95]",0
2,OdE7fBkHU5Q,2.0,2.0,2.0,200157.0,0.0,0.000000,1.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,/m/04rlf,441344.0,YT_OdE7fBkHU5Q.wav,20000,190.0,0.0,http://youtu.be/OdE7fBkHU5Q?start=190&end=200,"[113, 244, 366]",0
3,GiuXhEKPBog,3.0,3.0,3.0,200107.0,0.0,0.000000,1.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,,480256.0,YT_GiuXhEKPBog.wav,20000,100.0,0.0,http://youtu.be/GiuXhEKPBog?start=100&end=110,"[149, 379]",0
4,AWFMvRrhj_E,4.0,4.0,4.0,200534.0,0.0,0.000000,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,,481280.0,YT_AWFMvRrhj_E.wav,20000,130.0,0.0,http://youtu.be/AWFMvRrhj_E?start=130&end=140,"[379, 511]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
718620,FD3FP4LGnr8,742430.0,742430.0,736907.0,200157.0,0.0,0.000000,1.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,/m/04rlf,441344.0,YT_FD3FP4LGnr8.wav,20000,150.0,0.0,http://youtu.be/FD3FP4LGnr8?start=150&end=160,[153],0
718621,0n5h-em4llg,742431.0,742431.0,736908.0,200107.0,0.0,0.000000,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,,480256.0,YT_0n5h-em4llg.wav,20000,30.0,0.0,http://youtu.be/0n5h-em4llg?start=30&end=40,[486],0
718622,iLoINEceZmg,742432.0,742432.0,736909.0,200157.0,0.0,0.000000,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,,441344.0,YT_iLoINEceZmg.wav,20000,120.0,0.0,http://youtu.be/iLoINEceZmg?start=120&end=130,[0],0
718623,af_iEa_rViA,742433.0,742433.0,736910.0,200534.0,1005.0,0.005012,0.0,AUDIOSET,unbalanced_train_segments,...,/om2/data/public/audioset/wavs/unbalanced_trai...,,481280.0,YT_af_iEa_rViA.wav,20000,0.0,0.0,http://youtu.be/af_iEa_rViA?start=0&end=10,[37],0


In [28]:
all_audioset_labels = [l for label in noise_metadata.labels.values for l in label.split(',')]

In [29]:
np.unique(all_audioset_labels).shape

(516,)

: 